In [ ]:
# Read aws credentials from csv file and set as environment variables for Vault configuration
import csv, os
with open('vault_test_accessKeys.csv', 'r', encoding='utf-8-sig') as csvfile:
    reader = csv.DictReader(csvfile)
    creds = next(reader)
    access_key = creds['Access key ID']
    secret_key = creds['Secret access key']
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    print(f"Access Key: {access_key[:8]}...")
    print(f"Credentials loaded.")

# AWS Auth Method

In [ ]:
! open -a Podman\ Desktop

In [ ]:
%env VAULT_ADDR=http://127.0.0.1:8200
%env VAULT_TOKEN=root
%env VAULT_KMIP_PORT=5696
%env VAULT_PORT=8200
%env REGION=eu-west-3
%env VAULT_PUBLIC_ADDR=https://018d-88-11-147-194.ngrok-free.app

### Create a Vault Server in Podman

In [ ]:
%%bash
# Change the path to your license file

export VAULT_LICENSE=$(cat vault.hclic)

# Refresh Vault docker image with latest version
podman pull hashicorp/vault-enterprise

podman run -d --rm --name vault-enterprise --cap-add=IPC_LOCK \
  -e "VAULT_DEV_ROOT_TOKEN_ID=${VAULT_TOKEN}" \
  -e "VAULT_DEV_LISTEN_ADDRESS=:${VAULT_PORT}" \
  -e "VAULT_LICENSE=${VAULT_LICENSE}" \
  -p ${VAULT_KMIP_PORT}:${VAULT_KMIP_PORT} \
  -p 8200:${VAULT_PORT} \
  hashicorp/vault-enterprise:latest

### Check if Vault is running

In [ ]:
! podman ps

## Enable the AWS auth method

`vault auth enable aws` is run automatically as the first step of the **Configure AWS Auth Method Client Credentials** cell below — no need to run it separately.

### Create `vault-auth-root` IAM User

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query 'Account' --output text)
IAM_USER="vault-auth-root"

echo "Account ID: $ACCOUNT_ID"
echo "User:       $IAM_USER"

# Create IAM user (skip if already exists)
if aws iam get-user --user-name "$IAM_USER" &>/dev/null; then
    echo "IAM user $IAM_USER already exists, skipping creation."
else
    aws iam create-user --user-name "$IAM_USER"
    echo "IAM user created."
fi

# Recommended Vault IAM policy — as per:
# https://developer.hashicorp.com/vault/docs/auth/aws#recommended-vault-iam-policy
#
# Permissions breakdown:
#   ec2:DescribeInstances          — validate EC2 instances meet role binding requirements (ec2 auth / IAM inferencing)
#   iam:GetInstanceProfile         — resolve IAM role attached to an EC2 instance profile (bound_iam_role_arn on ec2 roles)
#   iam:GetUser / iam:GetRole      — resolve full ARN when a wildcard is used in bound_iam_principal_arn
#   sts:AssumeRole                 — cross-account access (Vault assumes a role in remote accounts to validate principals)
#   ManageOwnAccessKeys            — allow Vault to rotate its own static credentials via the rotate-root API
POLICY_NAME="VaultAuthMethodPolicy"
POLICY_ARN="arn:aws:iam::${ACCOUNT_ID}:policy/${POLICY_NAME}"

cat > /tmp/vault-auth-policy.json <<EOF
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "VaultAuthValidation",
      "Effect": "Allow",
      "Action": [
        "ec2:DescribeInstances",
        "iam:GetInstanceProfile",
        "iam:GetUser",
        "iam:GetRole"
      ],
      "Resource": "*"
    },
    {
      "Sid": "VaultCrossAccountAccess",
      "Effect": "Allow",
      "Action": ["sts:AssumeRole"],
      "Resource": ["arn:aws:iam::${ACCOUNT_ID}:role/*"]
    },
    {
      "Sid": "ManageOwnAccessKeys",
      "Effect": "Allow",
      "Action": [
        "iam:CreateAccessKey",
        "iam:DeleteAccessKey",
        "iam:GetAccessKeyLastUsed",
        "iam:GetUser",
        "iam:ListAccessKeys",
        "iam:UpdateAccessKey"
      ],
      "Resource": "arn:aws:iam::*:user/\${aws:username}"
    }
  ]
}
EOF

# Create or update the policy
if aws iam get-policy --policy-arn "$POLICY_ARN" &>/dev/null; then
    echo "Policy $POLICY_NAME already exists, updating to latest version..."
    # Remove oldest non-default version if at the 5-version AWS limit
    OLD_VERSION=$(aws iam list-policy-versions --policy-arn "$POLICY_ARN" \
        --query 'Versions[?IsDefaultVersion==`false`]|[-1].VersionId' --output text)
    if [ "$OLD_VERSION" != "None" ] && [ -n "$OLD_VERSION" ]; then
        aws iam delete-policy-version --policy-arn "$POLICY_ARN" --version-id "$OLD_VERSION" 2>/dev/null || true
    fi
    aws iam create-policy-version --policy-arn "$POLICY_ARN" \
        --policy-document file:///tmp/vault-auth-policy.json --set-as-default
    echo "Policy updated."
else
    aws iam create-policy --policy-name "$POLICY_NAME" \
        --policy-document file:///tmp/vault-auth-policy.json
    echo "Policy $POLICY_NAME created."
fi

# Detach any previously attached policies (e.g. old VaultIAMPolicy from Secrets Engine demo)
for OLD_ARN in $(aws iam list-attached-user-policies --user-name "$IAM_USER" \
        --query 'AttachedPolicies[*].PolicyArn' --output text 2>/dev/null); do
    if [ "$OLD_ARN" != "$POLICY_ARN" ]; then
        aws iam detach-user-policy --user-name "$IAM_USER" --policy-arn "$OLD_ARN"
        echo "Detached old policy: $OLD_ARN"
    fi
done

# Attach the correct policy
aws iam attach-user-policy --user-name "$IAM_USER" --policy-arn "$POLICY_ARN"
echo "Policy $POLICY_NAME attached to $IAM_USER."

echo ""
echo "=== Effective policies on $IAM_USER ==="
aws iam list-attached-user-policies --user-name "$IAM_USER" \
    --query 'AttachedPolicies[*].[PolicyName,PolicyArn]' --output table

# Delete all existing access keys (AWS limit is 2)
echo ""
echo "Cleaning up old access keys..."
for OLD_KEY in $(aws iam list-access-keys --user-name "$IAM_USER" \
        --query 'AccessKeyMetadata[*].AccessKeyId' --output text); do
    echo "  Deleting key: $OLD_KEY"
    aws iam delete-access-key --user-name "$IAM_USER" --access-key-id "$OLD_KEY"
done

# Create a fresh access key and persist it so the next cell can load it automatically
echo ""
echo "Creating new access key..."
KEY_JSON=$(aws iam create-access-key --user-name "$IAM_USER" \
    --query 'AccessKey.{KeyId:AccessKeyId,Secret:SecretAccessKey}' --output json)
KEY_ID=$(echo "$KEY_JSON"     | python3 -c "import json,sys; d=json.load(sys.stdin); print(d['KeyId'])")
KEY_SECRET=$(echo "$KEY_JSON" | python3 -c "import json,sys; d=json.load(sys.stdin); print(d['Secret'])")

# Save to file — the next cell (Python) reads this automatically; no manual copy-paste needed
echo "{\"access_key\": \"$KEY_ID\", \"secret_key\": \"$KEY_SECRET\"}" > /tmp/vault-iam-creds.json
chmod 600 /tmp/vault-iam-creds.json

echo "VAULT_IAM_ACCESS_KEY=$KEY_ID"
echo "VAULT_IAM_SECRET_KEY=<persisted to /tmp/vault-iam-creds.json>"
echo ""
echo "Run the next cell to activate these credentials in the kernel."


### Set `vault-root` Credentials
Copy the `VAULT_IAM_ACCESS_KEY` and `VAULT_IAM_SECRET_KEY` values from the output above.

In [ ]:
import json, os

# Reads the credentials written by the cell above — no manual copy-paste required.
# If the file is missing, re-run the cell above first.
with open("/tmp/vault-iam-creds.json") as f:
    creds = json.load(f)

access_key = creds["access_key"]
secret_key  = creds["secret_key"]

os.environ["VAULT_IAM_ACCESS_KEY"] = access_key
os.environ["VAULT_IAM_SECRET_KEY"]  = secret_key

print(f"VAULT_IAM_ACCESS_KEY = {access_key}")
print(f"VAULT_IAM_SECRET_KEY = {secret_key[:4]}...{secret_key[-4:]}  (truncated for display)")
print("Credentials loaded — ready to configure Vault.")


### Configure AWS Auth Method Client Credentials

Vault needs IAM credentials to call `sts:GetCallerIdentity` and validate incoming IAM auth requests.

In [ ]:
%%bash
# Enable the AWS auth method first (idempotent — safe to re-run)
vault auth enable aws 2>/dev/null && echo "AWS auth method enabled." || echo "AWS auth method already enabled."

# Configure Vault's AWS auth method with the vault-auth-root credentials.
# sts_endpoint MUST match the regional endpoint the Lambda signs against.
# Without it Vault rejects login requests whose iam_request_url points to a
# regional STS endpoint (e.g. sts.eu-west-3.amazonaws.com).
vault write auth/aws/config/client \
    access_key=$VAULT_IAM_ACCESS_KEY \
    secret_key=$VAULT_IAM_SECRET_KEY \
    region=$REGION \
    sts_endpoint=https://sts.${REGION}.amazonaws.com \
    sts_region=$REGION

echo ""
echo "=== AWS auth method client configuration ==="
vault read auth/aws/config/client


## Rotate root credentials

In [ ]:
! vault write -f /auth/aws/config/rotate-root

---
## AWS IAM Auth Method — Lambda Demo

This section tests the full end-to-end flow:
1. Create a static KV secret in Vault
2. Create a Vault policy granting read access to the secret
3. Create an AWS IAM execution role for Lambda
4. Configure a Vault role bound to the Lambda IAM role ARN
5. Deploy a Lambda function that authenticates to Vault via IAM and retrieves the secret

The Lambda function uses the IAM credentials automatically injected by AWS to sign a `sts:GetCallerIdentity` request, which Vault validates via STS before issuing a token.

Vault is reachable externally at: `https://018d-88-11-147-194.ngrok-free.app`

### Step 1 — Create a Static KV Secret in Vault

In [ ]:
%%bash
# Enable KV v2 secrets engine (skip if already enabled)
vault secrets enable -path=secret kv-v2 2>/dev/null && echo "KV v2 enabled at secret/" || echo "KV v2 already enabled."

# Create a static secret — simulates app credentials a Lambda might need
vault kv put secret/myapp/config \
    username="app-service-account" \
    password="Vault#S3cr3t!2026" \
    db_host="db.internal.example.com" \
    api_key="api-key-abc123xyz"

echo ""
echo "=== Secret stored at secret/myapp/config ==="
vault kv get secret/myapp/config

### Step 2 — Create a Vault Policy for Lambda

In [ ]:
%%bash
# Create a policy that only allows reading the static secret
vault policy write lambda-policy - <<'EOF'
# Allow Lambda to read the static app secret
path "secret/data/myapp/config" {
  capabilities = ["read"]
}
EOF

echo "=== Policy 'lambda-policy' created ==="
vault policy read lambda-policy

### Step 3 — Create an IAM Execution Role for Lambda

This role will be assumed by the Lambda function. Vault will be configured to trust this specific IAM role ARN.  
Copy the `LAMBDA_ROLE_ARN` value from the output and paste it in the next cell.

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query 'Account' --output text)
LAMBDA_ROLE_NAME="vault-lambda-demo-role"

echo "Account ID: $ACCOUNT_ID"
echo "Creating IAM role: $LAMBDA_ROLE_NAME"

# Trust policy — only Lambda service can assume this role
cat > /tmp/lambda-trust-policy.json <<EOF
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": { "Service": "lambda.amazonaws.com" },
      "Action": "sts:AssumeRole"
    }
  ]
}
EOF

# Create or skip if already exists
if aws iam get-role --role-name "$LAMBDA_ROLE_NAME" &>/dev/null; then
    echo "Role $LAMBDA_ROLE_NAME already exists, skipping creation."
else
    aws iam create-role \
        --role-name "$LAMBDA_ROLE_NAME" \
        --assume-role-policy-document file:///tmp/lambda-trust-policy.json \
        --description "Lambda execution role for Vault IAM auth demo"
    echo "Role created."
fi

# Attach the AWS-managed basic execution policy (CloudWatch Logs)
aws iam attach-role-policy \
    --role-name "$LAMBDA_ROLE_NAME" \
    --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole 2>/dev/null || true

ROLE_ARN=$(aws iam get-role --role-name "$LAMBDA_ROLE_NAME" --query 'Role.Arn' --output text)
echo ""
echo "LAMBDA_ROLE_ARN=$ROLE_ARN"

In [ ]:
%%bash
# Resolve the Lambda role ARN dynamically from AWS — no placeholder to update manually
LAMBDA_ROLE_ARN=$(aws iam get-role --role-name vault-lambda-demo-role \
    --query 'Role.Arn' --output text)
echo "LAMBDA_ROLE_ARN=$LAMBDA_ROLE_ARN"

### Step 4 — Configure a Vault Role Bound to the Lambda IAM Role

In [ ]:
%%bash
# Resolve the Lambda role ARN at runtime — avoids stale %env placeholders
LAMBDA_ROLE_ARN=$(aws iam get-role --role-name vault-lambda-demo-role \
    --query 'Role.Arn' --output text)

echo "Binding Vault role to: $LAMBDA_ROLE_ARN"

# Create a Vault role using IAM auth type, bound to the Lambda execution role ARN
# Only tokens obtained by that specific IAM role will be issued
vault write auth/aws/role/lambda-role \
    auth_type=iam \
    bound_iam_principal_arn="$LAMBDA_ROLE_ARN" \
    policies=lambda-policy \
    ttl=1h \
    max_ttl=4h

echo ""
echo "=== Vault role 'lambda-role' configured ==="
vault read auth/aws/role/lambda-role

### Step 5 — Package the Lambda Function

The function:
1. Reads its AWS credentials from the Lambda execution environment
2. Signs a `sts:GetCallerIdentity` POST request with SigV4
3. Sends the signed request components to Vault `/v1/auth/aws/login`
4. Uses the returned Vault token to read `secret/myapp/config`

In [ ]:
%%bash
mkdir -p /tmp/lambda-vault-demo

cat > /tmp/lambda-vault-demo/lambda_function.py <<'PYEOF'
import json
import os
import base64
import urllib.request
import urllib.error
import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest


def lambda_handler(event, context):
    vault_addr  = os.environ["VAULT_ADDR"].rstrip("/")
    vault_role  = os.environ.get("VAULT_ROLE", "lambda-role")
    secret_path = os.environ.get("SECRET_PATH", "secret/data/myapp/config")

    # ── 1. Get AWS credentials injected by the Lambda execution role ──────────
    session      = boto3.session.Session()
    frozen_creds = session.get_credentials().get_frozen_credentials()

    # Use the regional STS endpoint so the signing region matches the URL.
    # The global endpoint sts.amazonaws.com requires "us-east-1" in the
    # signature; using a regional endpoint avoids that mismatch.
    region   = os.environ.get("AWS_REGION", "us-east-1")
    sts_url  = f"https://sts.{region}.amazonaws.com/"
    sts_body = "Action=GetCallerIdentity&Version=2011-06-15"

    # ── 2. Build a signed sts:GetCallerIdentity request (Vault IAM auth) ──────
    aws_req = AWSRequest(
        method="POST",
        url=sts_url,
        data=sts_body,
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Content-Length": str(len(sts_body)),
        },
    )
    SigV4Auth(frozen_creds, "sts", region).add_auth(aws_req)

    signed_headers = dict(aws_req.headers)
    headers_json   = json.dumps({k: [v] for k, v in signed_headers.items()})

    # ── 3. Authenticate to Vault ───────────────────────────────────────────────
    vault_payload = {
        "role":                    vault_role,
        "iam_http_request_method": "POST",
        "iam_request_url":         base64.b64encode(sts_url.encode()).decode(),
        "iam_request_body":        base64.b64encode(sts_body.encode()).decode(),
        "iam_request_headers":     base64.b64encode(headers_json.encode()).decode(),
    }

    login_req = urllib.request.Request(
        f"{vault_addr}/v1/auth/aws/login",
        data=json.dumps(vault_payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(login_req, timeout=15) as resp:
            login_data = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        # Read and surface Vault's error body so it appears in CloudWatch logs
        err_body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Vault /auth/aws/login returned HTTP {e.code}. "
            f"STS URL signed against: {sts_url}  Region: {region}. "
            f"Vault error: {err_body}"
        )

    vault_token = login_data["auth"]["client_token"]
    token_ttl   = login_data["auth"]["lease_duration"]

    # ── 4. Read the static secret from Vault ─────────────────────────────────
    secret_req = urllib.request.Request(
        f"{vault_addr}/v1/{secret_path}",
        headers={"X-Vault-Token": vault_token},
        method="GET",
    )

    with urllib.request.urlopen(secret_req, timeout=15) as resp:
        secret_data = json.loads(resp.read())

    retrieved = secret_data.get("data", {}).get("data", {})

    return {
        "statusCode": 200,
        "body": json.dumps(
            {
                "message":              "Vault IAM authentication successful",
                "vault_role":           vault_role,
                "token_ttl_seconds":    token_ttl,
                "secret_keys_returned": list(retrieved.keys()),
                "username":             retrieved.get("username"),
                "db_host":              retrieved.get("db_host"),
            },
            indent=2,
        ),
    }
PYEOF

# Package
cd /tmp/lambda-vault-demo
zip -q lambda.zip lambda_function.py
echo "Lambda package ready: /tmp/lambda-vault-demo/lambda.zip ($(du -sh lambda.zip | cut -f1))"


### Step 6 — Deploy the Lambda Function

The function is configured with:
- `VAULT_ADDR` → ngrok public URL (reachable from Lambda in AWS)
- `VAULT_ROLE` → `lambda-role` (the Vault IAM auth role bound to the Lambda IAM role ARN)
- `SECRET_PATH` → `secret/data/myapp/config`

In [ ]:
import subprocess, os, json, time

FUNCTION_NAME  = "vault-aws-auth-demo"
region         = os.environ["REGION"]
vault_addr     = os.environ["VAULT_PUBLIC_ADDR"]

if not vault_addr:
    raise ValueError("VAULT_PUBLIC_ADDR is empty — re-run cell 4 first.")

# Resolve the Lambda execution role ARN from AWS
role_arn = subprocess.check_output(
    ["aws", "iam", "get-role", "--role-name", "vault-lambda-demo-role",
     "--query", "Role.Arn", "--output", "text"],
    text=True
).strip()

print(f"Deploying : {FUNCTION_NAME}")
print(f"Region    : {region}")
print(f"Role      : {role_arn}")
print(f"Vault     : {vault_addr}")

# Delete any existing version
subprocess.run(
    ["aws", "lambda", "delete-function", "--function-name", FUNCTION_NAME, "--region", region],
    capture_output=True
)
print("Deleted existing function (if any).")
time.sleep(10)  # IAM role propagation

env_config = {
    "Variables": {
        "VAULT_ADDR":  vault_addr,
        "VAULT_ROLE":  "lambda-role",
        "SECRET_PATH": "secret/data/myapp/config",
    }
}

result = subprocess.run(
    [
        "aws", "lambda", "create-function",
        "--function-name",  FUNCTION_NAME,
        "--runtime",        "python3.12",
        "--role",           role_arn,
        "--handler",        "lambda_function.lambda_handler",
        "--zip-file",       "fileb:///tmp/lambda-vault-demo/lambda.zip",
        "--timeout",        "30",
        "--region",         region,
        "--environment",    json.dumps(env_config),
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(f"create-function failed:\n{result.stderr}")

cfg = json.loads(result.stdout)
print("\n=== Lambda function deployed ===")
print(f"FunctionArn : {cfg['FunctionArn']}")
print(f"Runtime     : {cfg['Runtime']}")
print(f"State       : {cfg.get('State', 'n/a')}")
print(f"VAULT_ADDR  : {cfg['Environment']['Variables']['VAULT_ADDR']}")

### Step 7 — Invoke the Lambda Function and Verify

In [ ]:
%%bash
FUNCTION_NAME="vault-aws-auth-demo"
RESPONSE_FILE="/tmp/lambda-vault-response.json"

echo "Invoking $FUNCTION_NAME ..."
aws lambda invoke \
    --function-name "$FUNCTION_NAME" \
    --region "$REGION" \
    --payload '{}' \
    --cli-binary-format raw-in-base64-out \
    --log-type Tail \
    "$RESPONSE_FILE" \
    --query 'LogResult' \
    --output text 2>/dev/null | base64 --decode | tail -5

echo ""
echo "=== Lambda response ==="
cat "$RESPONSE_FILE" | python3 -c "
import json, sys
outer = json.load(sys.stdin)
print(f'Status: {outer[\"statusCode\"]}')
body = json.loads(outer['body'])
print(json.dumps(body, indent=2))
" 2>/dev/null || cat "$RESPONSE_FILE"

## Example with EC2 instance

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query 'Account' --output text)
EC2_ROLE_NAME="vault-ec2-demo-role"
PROFILE_NAME="vault-ec2-demo-profile"

echo "Account ID : $ACCOUNT_ID"

# ── 1. IAM role with EC2 service trust ───────────────────────────────────────
cat > /tmp/ec2-trust-policy.json <<EOF
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": { "Service": "ec2.amazonaws.com" },
    "Action": "sts:AssumeRole"
  }]
}
EOF

if aws iam get-role --role-name "$EC2_ROLE_NAME" &>/dev/null; then
    echo "IAM role $EC2_ROLE_NAME already exists, skipping."
else
    aws iam create-role \
        --role-name "$EC2_ROLE_NAME" \
        --assume-role-policy-document file:///tmp/ec2-trust-policy.json \
        --description "EC2 execution role for Vault AWS auth demo"
    echo "IAM role $EC2_ROLE_NAME created."
fi

# AmazonSSMManagedInstanceCore — lets us run SSM Run Command without opening SSH
aws iam attach-role-policy \
    --role-name "$EC2_ROLE_NAME" \
    --policy-arn arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore 2>/dev/null || true

# ── 2. Instance profile (required to attach an IAM role to an EC2 instance) ──
if aws iam get-instance-profile --instance-profile-name "$PROFILE_NAME" &>/dev/null; then
    echo "Instance profile $PROFILE_NAME already exists, skipping."
else
    aws iam create-instance-profile --instance-profile-name "$PROFILE_NAME"
    aws iam add-role-to-instance-profile \
        --instance-profile-name "$PROFILE_NAME" \
        --role-name "$EC2_ROLE_NAME"
    echo "Instance profile $PROFILE_NAME created and role attached."
fi

EC2_ROLE_ARN=$(aws iam get-role --role-name "$EC2_ROLE_NAME" --query 'Role.Arn' --output text)
echo ""
echo "EC2_ROLE_ARN=$EC2_ROLE_ARN"

# ── 3. Vault role using EC2 auth type ─────────────────────────────────────────
# auth_type=ec2: the instance submits its PKCS7-signed identity document.
# Vault verifies the AWS-issued signature and checks bound_iam_role_arn
# via ec2:DescribeInstances — no AWS credentials needed on the instance.
vault write auth/aws/role/ec2-role \
    auth_type=ec2 \
    bound_iam_role_arn="$EC2_ROLE_ARN" \
    policies=lambda-policy \
    disallow_reauthentication=false \
    ttl=1h \
    max_ttl=4h

echo ""
echo "=== Vault role 'ec2-role' ==="
vault read auth/aws/role/ec2-role


In [ ]:
%%bash
# vault-auth-check.sh — written to /tmp so the EC2 launch cell can embed it
# in user-data. The script runs automatically when the instance boots.

cat > /tmp/vault-auth-check.sh <<'SCRIPT_EOF'
#!/bin/bash
# vault-auth-check.sh
# Verifies Vault access from an EC2 instance using the AWS EC2 auth method.
#
# The EC2 instance identity document is signed by AWS (PKCS7 format).
# Vault verifies the signature offline and cross-checks the IAM role
# via ec2:DescribeInstances — no AWS credentials are needed on the instance.
#
# Required env vars:
#   VAULT_ADDR   — public Vault address (e.g. https://your-ngrok-url)
# Optional:
#   VAULT_ROLE   — Vault role name (default: ec2-role)
#   SECRET_PATH  — KV v2 path to read (default: secret/data/myapp/config)

set -euo pipefail

VAULT_ADDR="${VAULT_ADDR:?VAULT_ADDR must be set}"
VAULT_ROLE="${VAULT_ROLE:-ec2-role}"
SECRET_PATH="${SECRET_PATH:-secret/data/myapp/config}"

echo "============================================="
echo " Vault EC2 Auth Verification"
echo "============================================="
echo "Vault  : $VAULT_ADDR"
echo "Role   : $VAULT_ROLE"
echo "Secret : $SECRET_PATH"
echo "---------------------------------------------"

# 1. Acquire an IMDSv2 session token (required — IMDSv1 is disabled)
IMDS_TOKEN=$(curl -sf -X PUT "http://169.254.169.254/latest/api/token" \
    -H "X-aws-ec2-metadata-token-ttl-seconds: 60")
echo "[1] IMDSv2 session token acquired."

# 2. Read instance-level metadata (informational)
INSTANCE_ID=$(curl -sf -H "X-aws-ec2-metadata-token: $IMDS_TOKEN" \
    http://169.254.169.254/latest/meta-data/instance-id)
REGION=$(curl -sf -H "X-aws-ec2-metadata-token: $IMDS_TOKEN" \
    http://169.254.169.254/latest/meta-data/placement/region)
echo "[2] Instance : $INSTANCE_ID   Region: $REGION"

# 3. Fetch the PKCS7-signed EC2 identity document.
#    AWS signs this with its own private key; Vault calls ec2:DescribeInstances
#    and verifies the signature without needing credentials on the instance.
PKCS7=$(curl -sf -H "X-aws-ec2-metadata-token: $IMDS_TOKEN" \
    http://169.254.169.254/latest/dynamic/instance-identity/pkcs7 | tr -d '\n')
echo "[3] PKCS7 identity document retrieved (${#PKCS7} chars)."

# 4. Authenticate to Vault — POST the PKCS7 to /v1/auth/aws/login
echo "[4] Authenticating to Vault ..."
LOGIN_RESP=$(curl -sf -X POST "${VAULT_ADDR}/v1/auth/aws/login" \
    -H "Content-Type: application/json" \
    -d "{\"role\": \"${VAULT_ROLE}\", \"pkcs7\": \"${PKCS7}\"}")

VAULT_TOKEN=$(echo "$LOGIN_RESP" | python3 -c \
    "import json,sys; d=json.load(sys.stdin); print(d['auth']['client_token'])")
TOKEN_TTL=$(echo "$LOGIN_RESP" | python3 -c \
    "import json,sys; d=json.load(sys.stdin); print(d['auth']['lease_duration'])")
echo "[4] Vault token obtained. TTL: ${TOKEN_TTL}s"

# 5. Read the static secret with the Vault token
echo "[5] Reading secret from ${SECRET_PATH} ..."
SECRET_RESP=$(curl -sf \
    -H "X-Vault-Token: $VAULT_TOKEN" \
    "${VAULT_ADDR}/v1/${SECRET_PATH}")

echo "$SECRET_RESP" | python3 -c "
import json, sys
d = json.load(sys.stdin).get('data', {}).get('data', {})
print(f'[5] Secret keys : {list(d.keys())}')
print(f'    username    : {d.get(\"username\", \"(not set)\")}')
print(f'    db_host     : {d.get(\"db_host\",  \"(not set)\")}')
"

echo "============================================="
echo " All checks passed!"
echo "============================================="
SCRIPT_EOF

chmod +x /tmp/vault-auth-check.sh
echo "Script written to /tmp/vault-auth-check.sh"
echo ""
echo "--- Script content ---"
cat /tmp/vault-auth-check.sh


In [ ]:
import subprocess, os, json, base64

region     = os.environ["REGION"]
vault_addr = os.environ["VAULT_PUBLIC_ADDR"]

if not vault_addr:
    raise ValueError("VAULT_PUBLIC_ADDR is empty — re-run the env cell first.")

# Latest Amazon Linux 2023 AMI for the current region (via SSM public parameter)
ami_id = subprocess.check_output(
    ["aws", "ssm", "get-parameter",
     "--name", "/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-x86_64",
     "--region", region, "--query", "Parameter.Value", "--output", "text"],
    text=True
).strip()

profile_arn = subprocess.check_output(
    ["aws", "iam", "get-instance-profile",
     "--instance-profile-name", "vault-ec2-demo-profile",
     "--query", "InstanceProfile.Arn", "--output", "text"],
    text=True
).strip()

# Pick the first default subnet in the region (comes with a public IP)
subnet_id = subprocess.check_output(
    ["aws", "ec2", "describe-subnets",
     "--filters", "Name=default-for-az,Values=true",
     "--region", region,
     "--query", "Subnets[0].SubnetId", "--output", "text"],
    text=True
).strip()

# Read the verification script written in the previous cell
with open("/tmp/vault-auth-check.sh") as f:
    script_body = f.read()

# User-data: export Vault config vars, then run the script.
# Output lands in /var/log/cloud-init-output.log (read via SSM in the next cell).
user_data = f"""#!/bin/bash
export VAULT_ADDR="{vault_addr}"
export VAULT_ROLE="ec2-role"
export SECRET_PATH="secret/data/myapp/config"

{script_body}
"""

result = subprocess.run(
    [
        "aws", "ec2", "run-instances",
        "--image-id",           ami_id,
        "--instance-type",      "t3.micro",
        "--iam-instance-profile", f"Arn={profile_arn}",
        "--subnet-id",          subnet_id,
        "--user-data",          user_data,
        "--metadata-options",   "HttpTokens=required,HttpEndpoint=enabled",
        "--region",             region,
        "--tag-specifications",
        "ResourceType=instance,Tags=[{Key=Name,Value=vault-ec2-demo},{Key=Project,Value=vault-aws-auth-demo}]",
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

info        = json.loads(result.stdout)
instance_id = info["Instances"][0]["InstanceId"]

# Persist instance ID so the verify cell can pick it up without manual copy-paste
with open("/tmp/vault-ec2-instance-id.txt", "w") as f:
    f.write(instance_id)

print(f"Instance launched : {instance_id}")
print(f"AMI               : {ami_id}")
print(f"Region            : {region}")
print(f"Subnet            : {subnet_id}")
print(f"Instance profile  : {profile_arn}")
print(f"VAULT_ADDR        : {vault_addr}")
print()
print("vault-auth-check.sh will run automatically via user-data.")
print("Run the next cell (~2 min) to retrieve the output via SSM Run Command.")


In [ ]:
import subprocess, json, time, os

region = os.environ["REGION"]

with open("/tmp/vault-ec2-instance-id.txt") as f:
    instance_id = f.read().strip()

print(f"Waiting for {instance_id} to reach 'running' state ...")
subprocess.run(
    ["aws", "ec2", "wait", "instance-running",
     "--instance-ids", instance_id, "--region", region],
    check=True
)
print("Instance is running.")

# Give cloud-init / user-data time to execute before reading logs
print("Waiting 90 s for cloud-init to complete ...")
time.sleep(90)

# SSM Run Command — reads the cloud-init output log which contains the
# vault-auth-check.sh output (stdout + stderr of the user-data script)
print("Sending SSM Run Command ...")
send = subprocess.run(
    [
        "aws", "ssm", "send-command",
        "--instance-ids",    instance_id,
        "--document-name",   "AWS-RunShellScript",
        "--parameters",      json.dumps({"commands": ["cat /var/log/cloud-init-output.log"]}),
        "--region",          region,
        "--query",           "Command.CommandId",
        "--output",          "text",
    ],
    capture_output=True, text=True, check=True
)
cmd_id = send.stdout.strip()
print(f"Command ID : {cmd_id}")

# Poll until the command completes (usually < 10 s)
for _ in range(12):
    time.sleep(5)
    status = subprocess.run(
        ["aws", "ssm", "get-command-invocation",
         "--command-id", cmd_id, "--instance-id", instance_id,
         "--region", region, "--query", "Status", "--output", "text"],
        capture_output=True, text=True
    ).stdout.strip()
    if status in ("Success", "Failed", "Cancelled"):
        break
    print(f"  Status: {status} ...")

output = subprocess.run(
    ["aws", "ssm", "get-command-invocation",
     "--command-id", cmd_id, "--instance-id", instance_id,
     "--region", region, "--query", "StandardOutputContent", "--output", "text"],
    capture_output=True, text=True, check=True
).stdout

# Print from the first vault-auth-check line onwards
lines = output.splitlines()
start = next((i for i, l in enumerate(lines) if "Vault EC2 Auth" in l or "vault-auth-check" in l.lower()), 0)
print("\n" + "\n".join(lines[start:]))


# Clean up

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query 'Account' --output text)

# ── EC2 instance ──────────────────────────────────────────────────────────────
echo "=== Terminating EC2 instance ==="
if [ -f /tmp/vault-ec2-instance-id.txt ]; then
    INSTANCE_ID=$(cat /tmp/vault-ec2-instance-id.txt)
    aws ec2 terminate-instances --instance-ids "$INSTANCE_ID" --region "$REGION" 2>/dev/null \
        && echo "  Instance $INSTANCE_ID terminating." || echo "  Instance $INSTANCE_ID not found."
    rm -f /tmp/vault-ec2-instance-id.txt
else
    echo "  No instance ID file found, skipping."
fi

# ── EC2 IAM instance profile & role ──────────────────────────────────────────
echo ""
echo "=== Deleting EC2 IAM instance profile: vault-ec2-demo-profile ==="
EC2_PROFILE="vault-ec2-demo-profile"
EC2_ROLE="vault-ec2-demo-role"
aws iam remove-role-from-instance-profile \
    --instance-profile-name "$EC2_PROFILE" --role-name "$EC2_ROLE" 2>/dev/null || true
aws iam delete-instance-profile --instance-profile-name "$EC2_PROFILE" 2>/dev/null \
    && echo "  Instance profile $EC2_PROFILE deleted." || echo "  Instance profile $EC2_PROFILE not found."

echo ""
echo "=== Deleting IAM Role: $EC2_ROLE ==="
for POLICY_ARN in $(aws iam list-attached-role-policies --role-name "$EC2_ROLE" \
        --query 'AttachedPolicies[*].PolicyArn' --output text 2>/dev/null); do
    aws iam detach-role-policy --role-name "$EC2_ROLE" --policy-arn "$POLICY_ARN"
    echo "  Detached policy: $POLICY_ARN"
done
aws iam delete-role --role-name "$EC2_ROLE" 2>/dev/null \
    && echo "  Role $EC2_ROLE deleted." || echo "  Role $EC2_ROLE not found."

# ── Lambda function ───────────────────────────────────────────────────────────
echo ""
echo "=== Deleting Lambda function ==="
aws lambda delete-function --function-name "vault-aws-auth-demo" --region "$REGION" 2>/dev/null \
    && echo "  Lambda function deleted." || echo "  Lambda function not found."

echo ""
echo "=== Deleting IAM Role: vault-lambda-demo-role ==="
LAMBDA_ROLE="vault-lambda-demo-role"
for POLICY_ARN in $(aws iam list-attached-role-policies --role-name "$LAMBDA_ROLE" \
        --query 'AttachedPolicies[*].PolicyArn' --output text 2>/dev/null); do
    aws iam detach-role-policy --role-name "$LAMBDA_ROLE" --policy-arn "$POLICY_ARN"
    echo "  Detached policy: $POLICY_ARN"
done
aws iam delete-role --role-name "$LAMBDA_ROLE" 2>/dev/null \
    && echo "  Role $LAMBDA_ROLE deleted." || echo "  Role $LAMBDA_ROLE not found."

# ── IAM users ─────────────────────────────────────────────────────────────────
echo ""
echo "=== Deleting IAM Users ==="
for USER in vault-auth-root app-test vault-assumed test-assumed; do
    echo ""
    echo "--- Deleting user: $USER ---"

    for KEY in $(aws iam list-access-keys --user-name "$USER" \
            --query 'AccessKeyMetadata[*].AccessKeyId' --output text 2>/dev/null); do
        aws iam delete-access-key --user-name "$USER" --access-key-id "$KEY"
        echo "  Deleted access key: $KEY"
    done

    for POLICY_ARN in $(aws iam list-attached-user-policies --user-name "$USER" \
            --query 'AttachedPolicies[*].PolicyArn' --output text 2>/dev/null); do
        aws iam detach-user-policy --user-name "$USER" --policy-arn "$POLICY_ARN"
        echo "  Detached policy: $POLICY_ARN"
    done

    for POLICY_NAME in $(aws iam list-user-policies --user-name "$USER" \
            --query 'PolicyNames[*]' --output text 2>/dev/null); do
        aws iam delete-user-policy --user-name "$USER" --policy-name "$POLICY_NAME"
        echo "  Deleted inline policy: $POLICY_NAME"
    done

    aws iam delete-user --user-name "$USER" 2>/dev/null \
        && echo "  User $USER deleted." || echo "  User $USER not found or already deleted."
done

echo ""
echo "=== AWS cleanup complete ==="


In [ ]:
%%bash
echo "=== Stopping Vault container ==="
podman stop vault-enterprise 2>/dev/null && echo "Vault container stopped." || echo "Vault container not running."
echo "=== All demo resources cleaned up ==="